<a href="https://colab.research.google.com/github/exyde01-sketch/Artefactos/blob/main/Menu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
# ============================================================
# EQUIPO 6: EXTENDED INFOMAX ICA
# Algoritmo para Google Colab
# ============================================================

# ============================================================
# 1. INSTALACIÓN DE DEPENDENCIAS
# ============================================================

import importlib.util
import subprocess
import sys

dependencias = {
    "mne": "mne",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib"
}

for modulo, paquete in dependencias.items():
    if importlib.util.find_spec(modulo) is None:
        print(f"Instalando dependencia: {paquete}")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", paquete
        ])


# ============================================================
# 2. IMPORTACIONES
# ============================================================

import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

from mne.preprocessing import ICA
from IPython.display import display


# ============================================================
# 3. CONFIGURACIÓN
# ============================================================

# Cambiar por la frecuencia de muestreo REAL de los datos EEG.
FRECUENCIA_MUESTREO = 256.0

# Umbral heurístico para marcar componentes atípicos.
UMBRAL_KURTOSIS = 5.0

# Máximo de canales mostrados en la gráfica comparativa.
MAX_CANALES_COMPARACION = 4

# Cantidad máxima de iteraciones de ICA.
MAX_ITERACIONES_ICA = 1000


# ============================================================
# 4. FUNCIONES AUXILIARES
# ============================================================

def calcular_curtosis(componente):
    """
    Calcula la curtosis excedente de un componente independiente.
    """

    componente = np.asarray(componente, dtype=float)

    desviacion = np.std(componente)

    if desviacion < 1e-12:
        return 0.0

    normalizado = (
        componente - np.mean(componente)
    ) / desviacion

    return float(np.mean(normalizado ** 4) - 3.0)


def preparar_datos_csv(ruta_csv):
    """
    Lee y prepara el CSV.

    - Identifica una posible columna temporal.
    - Convierte las columnas de señales a valores numéricos.
    - Ignora columnas sin datos numéricos.
    - Trata valores ausentes mediante interpolación.
    - Comprueba que existan al menos dos canales válidos.

    Devuelve:
        df_original, datos, nombres_canales, columna_temporal, tiempo
    """

    df_original = pd.read_csv(ruta_csv)

    if df_original.empty:
        raise ValueError("El archivo CSV está vacío.")

    print(f"Filas encontradas: {len(df_original)}")
    print(f"Columnas encontradas: {list(df_original.columns)}")

    # Detectar una posible columna temporal.
    columnas_tiempo = {
        "timestamp",
        "time",
        "datetime",
        "fecha",
        "tiempo",
        "date"
    }

    columna_temporal = next(
        (
            columna
            for columna in df_original.columns
            if str(columna).strip().lower() in columnas_tiempo
        ),
        None
    )

    if columna_temporal is not None:
        tiempo = df_original[columna_temporal].copy()
    else:
        tiempo = None

    # Excluir la columna temporal de las señales.
    columnas_senales = [
        columna
        for columna in df_original.columns
        if columna != columna_temporal
    ]

    # Convertir las columnas a formato numérico.
    senales_df = df_original[columnas_senales].apply(
        pd.to_numeric,
        errors="coerce"
    )

    # Ignorar columnas que no contengan ningún dato numérico.
    columnas_vacias = senales_df.columns[
        senales_df.isna().all()
    ].tolist()

    if columnas_vacias:
        print(
            "Columnas ignoradas por no contener datos numéricos:",
            columnas_vacias
        )

        senales_df = senales_df.drop(
            columns=columnas_vacias
        )

    if senales_df.shape[1] < 2:
        raise ValueError(
            "Se necesitan al menos dos canales numéricos. "
            "Comprueba las columnas del archivo."
        )

    # Convertir infinitos en valores ausentes.
    senales_df = senales_df.replace(
        [np.inf, -np.inf],
        np.nan
    )

    # Informar de los valores ausentes.
    faltantes = int(senales_df.isna().sum().sum())

    if faltantes > 0:
        print(
            f"Se encontraron {faltantes} valores ausentes "
            "en las señales."
        )

        # Interpolar huecos usando las muestras vecinas.
        senales_df = senales_df.interpolate(
            method="linear",
            axis=0,
            limit_direction="both"
        )

        # No se pueden recuperar columnas totalmente vacías.
        columnas_sin_datos = senales_df.columns[
            senales_df.isna().all()
        ].tolist()

        if columnas_sin_datos:
            raise ValueError(
                "Estas columnas no contienen datos válidos: "
                + ", ".join(map(str, columnas_sin_datos))
            )

        # Comprobar que la interpolación haya sido suficiente.
        if senales_df.isna().any().any():
            raise ValueError(
                "Persisten valores ausentes después de la "
                "interpolación. Revisa el archivo original."
            )

        print("Valores ausentes tratados mediante interpolación.")

    datos = senales_df.to_numpy(dtype=float)
    nombres_canales = senales_df.columns.tolist()

    if not np.isfinite(datos).all():
        raise ValueError(
            "Persisten valores no finitos en las señales."
        )

    if len(datos) < 10:
        raise ValueError(
            "El archivo contiene muy pocas muestras para ICA."
        )

    # Verificar canales constantes.
    desviaciones = np.std(datos, axis=0)

    canales_constantes = [
        nombres_canales[i]
        for i, desviacion in enumerate(desviaciones)
        if desviacion < 1e-12
    ]

    if canales_constantes:
        raise ValueError(
            "Se detectaron canales constantes: "
            + ", ".join(map(str, canales_constantes))
            + ". Revisa o elimina esos canales."
        )

    return (
        df_original,
        datos,
        nombres_canales,
        columna_temporal,
        tiempo
    )


# ============================================================
# 5. PROCESAMIENTO DE UN ARCHIVO
# ============================================================

def procesar_archivo_eeg(
    ruta_csv,
    carpeta_salida,
    sfreq=FRECUENCIA_MUESTREO,
    umbral_kurtosis=UMBRAL_KURTOSIS
):
    """
    Procesa un CSV con Extended Infomax ICA.

    Genera:
        - CSV procesado.
        - Gráfica de curtosis.
        - Gráfica comparativa.
        - Resumen del procesamiento.
    """

    ruta_csv = Path(ruta_csv)
    carpeta_salida = Path(carpeta_salida)

    carpeta_salida.mkdir(
        parents=True,
        exist_ok=True
    )

    nombre_archivo = ruta_csv.stem

    print("\n" + "=" * 60)
    print(f"Procesando archivo: {ruta_csv.name}")
    print("=" * 60)

    # --------------------------------------------------------
    # A. LEER Y VALIDAR EL CSV
    # --------------------------------------------------------

    (
        df_original,
        datos_originales,
        nombres_canales,
        columna_temporal,
        tiempo
    ) = preparar_datos_csv(ruta_csv)

    cantidad_muestras = datos_originales.shape[0]
    cantidad_canales = datos_originales.shape[1]

    if not np.isfinite(sfreq) or sfreq <= 0:
        raise ValueError(
            "La frecuencia de muestreo debe ser positiva."
        )

    # --------------------------------------------------------
    # B. NORMALIZAR LOS CANALES
    # --------------------------------------------------------

    medias = np.mean(datos_originales, axis=0)
    escalas = np.std(datos_originales, axis=0)

    datos_normalizados = (
        datos_originales - medias
    ) / escalas

    # --------------------------------------------------------
    # C. CREAR EL OBJETO RAW DE MNE
    # --------------------------------------------------------

    info = mne.create_info(
        ch_names=[str(nombre) for nombre in nombres_canales],
        sfreq=sfreq,
        ch_types=["eeg"] * cantidad_canales
    )

    raw = mne.io.RawArray(
        datos_normalizados.T,
        info,
        verbose=False
    )

    # --------------------------------------------------------
    # D. EJECUTAR EXTENDED INFOMAX ICA
    # --------------------------------------------------------

    print("\nEjecutando Extended Infomax ICA...")

    ica = ICA(
        n_components=cantidad_canales,
        method="infomax",
        fit_params={"extended": True},
        random_state=42,
        max_iter=MAX_ITERACIONES_ICA,
        verbose=False
    )

    ica.fit(
        raw,
        picks="eeg",
        verbose=False
    )

    print("ICA finalizado.")

    # --------------------------------------------------------
    # E. OBTENER LOS COMPONENTES INDEPENDIENTES
    # --------------------------------------------------------

    fuentes = ica.get_sources(
        raw
    ).get_data()

    valores_curtosis = np.array([
        calcular_curtosis(fuentes[i])
        for i in range(fuentes.shape[0])
    ])

    # --------------------------------------------------------
    # F. DETECTAR POSIBLES ARTEFACTOS
    # --------------------------------------------------------

    componentes_excluidos = [
        i
        for i, valor in enumerate(valores_curtosis)
        if abs(valor) > umbral_kurtosis
    ]

    print(f"Canales analizados: {cantidad_canales}")
    print(f"Muestras analizadas: {cantidad_muestras}")
    print(f"Componentes ICA: {len(valores_curtosis)}")
    print(
        "Componentes marcados por curtosis:",
        len(componentes_excluidos)
    )

    if componentes_excluidos:
        print(
            "Componentes marcados (numeración desde 1):",
            [i + 1 for i in componentes_excluidos]
        )
    else:
        print("No se marcaron componentes con el umbral establecido.")

    # --------------------------------------------------------
    # G. RECONSTRUIR LAS SEÑALES
    # --------------------------------------------------------

    # La API de MNE aplica la reconstrucción usando la matriz
    # de mezcla y el centrado interno de ICA.
    raw_limpio = raw.copy()

    ica.exclude = componentes_excluidos

    ica.apply(
        raw_limpio,
        verbose=False
    )

    datos_limpios_normalizados = (
        raw_limpio.get_data().T
    )

    # Deshacer la normalización.
    datos_limpios = (
        datos_limpios_normalizados * escalas
    ) + medias

    # --------------------------------------------------------
    # H. GUARDAR CSV PROCESADO
    # --------------------------------------------------------

    df_limpio = pd.DataFrame(
        datos_limpios,
        columns=nombres_canales
    )

    # Conservar la columna temporal, si existía.
    if columna_temporal is not None:
        df_limpio.insert(
            0,
            columna_temporal,
            tiempo.reset_index(drop=True)
        )

    ruta_csv_limpio = (
        carpeta_salida
        / f"{nombre_archivo}_limpio.csv"
    )

    df_limpio.to_csv(
        ruta_csv_limpio,
        index=False
    )

    print(f"CSV guardado: {ruta_csv_limpio.name}")

    # --------------------------------------------------------
    # I. GRÁFICA DE CURTOSIS
    # --------------------------------------------------------

    ruta_grafica_curtosis = (
        carpeta_salida
        / f"{nombre_archivo}_deteccion.png"
    )

    indices = np.arange(
        1,
        len(valores_curtosis) + 1
    )

    colores = [
        "red" if i in componentes_excluidos else "steelblue"
        for i in range(len(valores_curtosis))
    ]

    fig, ax = plt.subplots(figsize=(12, 5))

    ax.bar(
        indices,
        valores_curtosis,
        color=colores
    )

    ax.axhline(
        umbral_kurtosis,
        linestyle="--",
        color="black",
        label=f"Umbral +{umbral_kurtosis}"
    )

    ax.axhline(
        -umbral_kurtosis,
        linestyle="--",
        color="gray",
        label=f"Umbral -{umbral_kurtosis}"
    )

    ax.set_title(
        f"Extended Infomax ICA: {nombre_archivo}"
    )

    ax.set_xlabel("Componente independiente")
    ax.set_ylabel("Curtosis excedente")
    ax.set_xticks(indices)
    ax.grid(axis="y", alpha=0.25)
    ax.legend()

    fig.tight_layout()

    fig.savefig(
        ruta_grafica_curtosis,
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    # --------------------------------------------------------
    # J. GRÁFICA COMPARATIVA
    # --------------------------------------------------------

    ruta_grafica_comparacion = (
        carpeta_salida
        / f"{nombre_archivo}_comparativa.png"
    )

    cantidad_graficas = min(
        MAX_CANALES_COMPARACION,
        cantidad_canales
    )

    tiempo_segundos = (
        np.arange(cantidad_muestras) / sfreq
    )

    fig, ejes = plt.subplots(
        cantidad_graficas,
        1,
        figsize=(14, 3 * cantidad_graficas),
        squeeze=False
    )

    for i in range(cantidad_graficas):

        ax = ejes[i, 0]

        ax.plot(
            tiempo_segundos,
            datos_originales[:, i],
            label="Original",
            linewidth=0.8,
            alpha=0.8
        )

        ax.plot(
            tiempo_segundos,
            datos_limpios[:, i],
            label="Procesada",
            linewidth=0.8,
            alpha=0.8
        )

        ax.set_title(
            f"Canal: {nombres_canales[i]}"
        )

        ax.set_xlabel("Tiempo (s)")
        ax.set_ylabel("Amplitud")
        ax.grid(True, alpha=0.25)
        ax.legend()

    fig.suptitle(
        f"Señal original vs. procesada: {nombre_archivo}"
    )

    fig.tight_layout()

    fig.savefig(
        ruta_grafica_comparacion,
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    # --------------------------------------------------------
    # K. RESUMEN DEL ARCHIVO
    # --------------------------------------------------------

    resumen = {
        "archivo": ruta_csv.name,
        "muestras": cantidad_muestras,
        "canales": cantidad_canales,
        "componentes_ICA": len(valores_curtosis),
        "componentes_marcados": len(componentes_excluidos),
        "componentes_excluidos": ", ".join(
            str(i + 1)
            for i in componentes_excluidos
        ),
        "csv_procesado": ruta_csv_limpio.name,
        "grafica_curtosis": ruta_grafica_curtosis.name,
        "grafica_comparativa": ruta_grafica_comparacion.name
    }

    print(f"Finalizado: {ruta_csv.name}")

    return resumen


# ============================================================
# 6. FUNCIÓN QUE LLAMA EL MENÚ
# ============================================================

def ejecutar_equipo6():

    try:
        from google.colab import files
    except ImportError:
        print(
            "Error: esta función debe ejecutarse en Google Colab."
        )
        return

    print("\n" + "=" * 55)
    print("EQUIPO 6 — EXTENDED INFOMAX ICA")
    print("=" * 55)

    print(
        "\nSelecciona uno o varios archivos CSV con señales EEG."
    )

    archivos_subidos = files.upload()

    if not archivos_subidos:
        print("No se seleccionaron archivos.")
        return

    carpeta_entrada = Path("/content/entrada_eeg")
    carpeta_salida = Path("/content/resultados_eeg")

    carpeta_entrada.mkdir(
        parents=True,
        exist_ok=True
    )

    # Crear una carpeta nueva para los resultados de esta ejecución.
    if carpeta_salida.exists():
        shutil.rmtree(carpeta_salida)

    carpeta_salida.mkdir(
        parents=True,
        exist_ok=True
    )

    rutas_csv = []

    # Guardar únicamente archivos CSV.
    for nombre, contenido in archivos_subidos.items():

        if not nombre.lower().endswith(".csv"):
            print(f"Archivo ignorado porque no es CSV: {nombre}")
            continue

        ruta = carpeta_entrada / Path(nombre).name

        with open(ruta, "wb") as archivo:
            archivo.write(contenido)

        rutas_csv.append(ruta)

    if not rutas_csv:
        print("No se encontraron archivos CSV válidos.")
        return

    resultados = []
    errores = []

    # Procesar los archivos individualmente.
    for ruta in rutas_csv:

        try:
            resumen = procesar_archivo_eeg(
                ruta_csv=ruta,
                carpeta_salida=carpeta_salida,
                sfreq=FRECUENCIA_MUESTREO,
                umbral_kurtosis=UMBRAL_KURTOSIS
            )

            resultados.append(resumen)

        except Exception as error:

            print(
                f"\nError al procesar {ruta.name}: {error}"
            )

            errores.append({
                "archivo": ruta.name,
                "error": str(error)
            })

    # --------------------------------------------------------
    # 7. GENERAR RESÚMENES
    # --------------------------------------------------------

    if resultados:

        df_resumen = pd.DataFrame(resultados)

        ruta_resumen = (
            carpeta_salida
            / "resumen_procesamiento.csv"
        )

        df_resumen.to_csv(
            ruta_resumen,
            index=False
        )

        print("\nRESUMEN DE PROCESAMIENTO")
        display(df_resumen)

    if errores:

        df_errores = pd.DataFrame(errores)

        ruta_errores = (
            carpeta_salida
            / "errores_procesamiento.csv"
        )

        df_errores.to_csv(
            ruta_errores,
            index=False
        )

        print("\nARCHIVOS CON ERRORES")
        display(df_errores)

    # --------------------------------------------------------
    # 8. EMPAQUETAR Y DESCARGAR RESULTADOS
    # --------------------------------------------------------

    archivos_resultado = [
        archivo
        for archivo in carpeta_salida.rglob("*")
        if archivo.is_file()
    ]

    if not archivos_resultado:
        print(
            "\nNo se generaron archivos para descargar. "
            "Revisa los errores anteriores."
        )
        return

    ruta_zip = Path("/content/resultados_eeg.zip")

    if ruta_zip.exists():
        ruta_zip.unlink()

    with zipfile.ZipFile(
        ruta_zip,
        "w",
        compression=zipfile.ZIP_DEFLATED
    ) as archivo_zip:

        for archivo in archivos_resultado:

            archivo_zip.write(
                archivo,
                arcname=archivo.relative_to(carpeta_salida)
            )

    print("\n" + "=" * 55)
    print("PROCESAMIENTO FINALIZADO")
    print("=" * 55)
    print(f"Archivos procesados: {len(resultados)}")
    print(f"Archivos con errores: {len(errores)}")
    print("Descargando CSV y gráficas en un archivo ZIP...")

    files.download(str(ruta_zip))

In [16]:

def mostrar_menu():
    print("\n" + "=" * 50)
    print("       MENÚ DE PROCESAMIENTO DE SEÑALES")
    print("=" * 50)
    print("1. Equipo 1 - Fast_ICA")
    print("2. Equipo 2 - Wavelet_ICA")
    print("3. Equipo 3 - Pasaband")
    print("4. Equipo 4 - InterferenciaModifier")
    print("5. Equipo 5 - ICA")
    print("6. Equipo 6 - ExtendInfomax_ICA")
    print("7. Equipo 7 - Artefacto_NoRaw")
    print("0. Salir")
    print("=" * 50)


def ejecutar_equipo(opcion):
    temas = {
        "1": ("Equipo 1", "Fast_ICA"),
        "2": ("Equipo 2", "Wavelet_ICA"),
        "3": ("Equipo 3", "Pasaband"),
        "4": ("Equipo 4", "InterferenciaModifier"),
        "5": ("Equipo 5", "ICA"),
        "6": ("Equipo 6", "ExtendInfomax_ICA"),
        "7": ("Equipo 7", "Artefacto_NoRaw")
    }

    if opcion in temas:
        equipo, tema = temas[opcion]
        print("\n" + "-" * 40)
        print(f"Seleccionaste: {equipo}")
        print(f"Tema: {tema}")
        print("-" * 40)

        # Aquí puedes agregar el código específico
        # para ejecutar el procesamiento de cada tema.
        # Implementación de la opción 6
        if opcion == "6":
            ejecutar_equipo6()


    else:
        print("Opción no válida. Intenta nuevamente.")


def main():
    while True:
        mostrar_menu()
        opcion = input("Selecciona una opción (0-7): ").strip()

        if opcion == "0":
            print("Saliendo del programa. ¡Hasta pronto!")
            break

        ejecutar_equipo(opcion)


if __name__ == "__main__":
    main()



       MENÚ DE PROCESAMIENTO DE SEÑALES
1. Equipo 1 - Fast_ICA
2. Equipo 2 - Wavelet_ICA
3. Equipo 3 - Pasaband
4. Equipo 4 - InterferenciaModifier
5. Equipo 5 - ICA
6. Equipo 6 - ExtendInfomax_ICA
7. Equipo 7 - Artefacto_NoRaw
0. Salir
Selecciona una opción (0-7): 0
Saliendo del programa. ¡Hasta pronto!
